In [2]:
# ===============================================
# CONTOH SEDERHANA
# DETEKSI FACE + HAND + POSE LANDMARK
# Tekan Q untuk keluar
# ===============================================

import cv2
import time
import mediapipe as mp

# ===============================================
# MEDIAPIPE HOLISTIC
# (face + hand + pose sekaligus)
# ===============================================
mp_holistic = mp.solutions.holistic
mp_draw = mp.solutions.drawing_utils

holistic = mp_holistic.Holistic(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# ===============================================
# CAMERA
# ===============================================
cap = cv2.VideoCapture(0)

prev_time = 0

while cap.isOpened():

    ret, frame = cap.read()

    if not ret:
        break

    # mirror camera
    frame = cv2.flip(frame, 1)

    # resize
    frame = cv2.resize(frame, (900, 650))

    # BGR -> RGB
    image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # process
    results = holistic.process(image)

    # RGB -> BGR
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

    # ===============================================
    # FACE LANDMARK
    # ===============================================
    mp_draw.draw_landmarks(
        image,
        results.face_landmarks,
        mp_holistic.FACEMESH_CONTOURS,
        mp_draw.DrawingSpec(color=(255,0,255), thickness=1, circle_radius=1),
        mp_draw.DrawingSpec(color=(0,255,255), thickness=1, circle_radius=1)
    )

    # ===============================================
    # LEFT HAND
    # ===============================================
    mp_draw.draw_landmarks(
        image,
        results.left_hand_landmarks,
        mp_holistic.HAND_CONNECTIONS,
        mp_draw.DrawingSpec(color=(0,255,0), thickness=2, circle_radius=2),
        mp_draw.DrawingSpec(color=(255,255,255), thickness=2, circle_radius=2)
    )

    # ===============================================
    # RIGHT HAND
    # ===============================================
    mp_draw.draw_landmarks(
        image,
        results.right_hand_landmarks,
        mp_holistic.HAND_CONNECTIONS,
        mp_draw.DrawingSpec(color=(0,0,255), thickness=2, circle_radius=2),
        mp_draw.DrawingSpec(color=(255,255,255), thickness=2, circle_radius=2)
    )

    # ===============================================
    # BODY / POSE LANDMARK
    # ===============================================
    mp_draw.draw_landmarks(
        image,
        results.pose_landmarks,
        mp_holistic.POSE_CONNECTIONS,
        mp_draw.DrawingSpec(color=(255,255,0), thickness=2, circle_radius=2),
        mp_draw.DrawingSpec(color=(0,255,0), thickness=2, circle_radius=2)
    )

    # ===============================================
    # FPS
    # ===============================================
    current_time = time.time()
    fps = 1 / (current_time - prev_time) if prev_time != 0 else 0
    prev_time = current_time

    cv2.putText(image, f"FPS : {int(fps)}",
                (20,40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1, (0,255,0), 2)

    cv2.putText(image, "FACE + HAND + POSE",
                (20,80),
                cv2.FONT_HERSHEY_SIMPLEX,
                1, (255,255,255), 2)

    cv2.putText(image, "Press Q to Exit",
                (20,620),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7, (255,255,255), 2)

    # ===============================================
    # SHOW
    # ===============================================
    cv2.imshow("Holistic Landmark Detection", image)

    if cv2.waitKey(5) & 0xFF == ord('q'):
        break

# ===============================================
# CLOSE
# ===============================================
cap.release()
cv2.destroyAllWindows()